In [1]:
import math
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
from matplotlib.axes import Axes
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from plotting_defaults import COLOR_MAP, delete_tex_cache

from fedaugment.utils import LogParser, cd_to_git_root

cd_to_git_root()
delete_tex_cache()

plots_dir = Path("analysis/talk")
plots_dir.mkdir(exist_ok=True, parents=True)

2026-08-27 17:45:49.655 | DEBUG    | fedaugment.utils.misc:cd_to_git_root:154 - Changed working directory to: /home/lbhm/Projects/fedaugment


In [2]:
sns.set_theme(
    context="talk",
    style="ticks",
    palette="colorblind",
    color_codes=True,
    font_scale=1,
    rc={
        # Generic settings #
        "backend": "svg",
        "font.family": "Aptos",
        "savefig.bbox": "tight",  # {tight, standard}
        "savefig.format": "svg",  # {svg}
        "savefig.pad_inches": 0.0075,  # padding to be used, when bbox is set to 'tight'
        "savefig.transparent": True,  # transparent background
        "svg.fonttype": "none",  # {none, path, svgfont}
        # Layout settings #
        "axes.labelsize": 12,
        "hatch.linewidth": 1.5,
        "legend.columnspacing": 1.5,  # 2.0
        "legend.fontsize": 12,  # medium
        "legend.frameon": False,  # True
        "legend.handletextpad": 0.6,  # 0.8
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    },
)

## Curation Figure

In [3]:
log_dir = Path("logs/curation/webtable")

parser = LogParser()
logs = parser.parse(log_dir / "train_2026-04-16_16-46-29.log")
logs.extend(parser.parse(log_dir / "train_2026-04-20_11-56-41.log"))
logs.extend(parser.parse(log_dir / "train_2026-04-20_11-57-50.log"))
logs.extend(parser.parse(log_dir / "train_2026-04-20_11-57-58.log"))

data = defaultdict(dict)
for log in logs:
    if log["config"]:
        exp_name: str = log["config"]["experiment_name"].replace("webtable_", "")
        exp_config = exp_name.split("-")
        method = exp_config[0]
        if len(exp_config) > 1:
            n_samples = int(exp_config[3][2:])
            sampling_ratio = (
                0.05 if n_samples == 630493 else 0.005 if n_samples == 63049 else 0.001
            )
        else:
            sampling_ratio = 1.0
        with_pca = "pca" in exp_name

        if with_pca:
            method += "-pca"
        if "630493" in exp_name:
            data["0.05"][method] = log["test_metrics"]["entity_stability_50"].item()
        elif "63049" in exp_name:
            data["0.005"][method] = log["test_metrics"]["entity_stability_50"].item()
        elif "12609" in exp_name:
            data["0.001"][method] = log["test_metrics"]["entity_stability_50"].item()
        elif "full" in exp_name:
            data["1"]["full"] = log["test_metrics"]["entity_stability_50"].item()

In [4]:
fig, ax = plt.subplots(figsize=(8, 3.7), layout="tight")

current_x = 0
group_gap = 1
xticks: list[float] = []
xticklabels: list[str] = []

for sampling_rate, methods in sorted(data.items()):
    ordered_methods = dict(sorted(methods.items()))
    positions = [current_x + idx for idx in range(len(ordered_methods))]
    xticks.append(positions[0] + (positions[-1] - positions[0]) / 2)
    xticklabels.append(sampling_rate)

    for pos, (method_name, value) in zip(positions, ordered_methods.items(), strict=True):
        base_method = method_name.replace("-pca", "")
        color = COLOR_MAP[base_method]
        is_pca = method_name.endswith("-pca")
        hatch = "xxx" if is_pca else None

        bar = ax.bar(pos, value, color=color, hatch=hatch, edgecolor="black")
    current_x += len(ordered_methods) + group_gap

legend_handles = [
    Patch(
        facecolor=COLOR_MAP["fft_cos"],
        edgecolor="black",
        label="Farthest-first traversal (cosine similarity)",
    ),
    Patch(
        facecolor=COLOR_MAP["fft_euc"],
        edgecolor="black",
        label="Farthest-first traversal (Euclidean distance)",
    ),
    Patch(facecolor=COLOR_MAP["grid"], edgecolor="black", label="Grid-based sampling"),
    Patch(facecolor=COLOR_MAP["random"], edgecolor="black", label="Random sampling"),
    Patch(facecolor=COLOR_MAP["full"], edgecolor="black", label="Full train split"),
    Patch(facecolor="white", edgecolor="black", hatch="xxxxx", label="PCA variant"),
]

ax.set_xlabel("Sampling rate")
ax.set_ylabel("Entity stability @ 50")
ax.set_xlim(-0.75, current_x - group_gap - 0.25)
ax.set_ylim(0, 1)
ax.set_xticks(xticks, xticklabels)
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.legend(handles=legend_handles, frameon=False, ncol=1)

fig.savefig(plots_dir / "curation.svg")
plt.close(fig)

## Effectiveness Figure

In [5]:
# Load results per dataset and experiment group
effectiveness_path = Path("logs/augmentations")

# Dict: dataset -> experiment_group -> {model_name: df}
results_by_experiment: dict[str, dict[str, dict[str, pl.DataFrame]]] = {}
for dataset_path in effectiveness_path.iterdir():
    if not dataset_path.is_dir():
        continue
    dataset = dataset_path.name
    results_by_experiment[dataset] = defaultdict(dict)

    for exp_group_path in dataset_path.iterdir():
        if not exp_group_path.is_dir():
            continue
        exp_group = exp_group_path.name

        for f in exp_group_path.glob("*.csv"):
            if exp_group == "centralized":
                model_name = f.stem.replace("centralized_", "").replace("-dj_adpt", "")
            else:
                model_name = f.stem.split("-")[1]
            results_by_experiment[dataset][exp_group][model_name] = pl.read_csv(f)

    expgroup_str = ", ".join(f"{k}({len(v)})" for k, v in results_by_experiment[dataset].items())
    print(f"{dataset}: {expgroup_str}")

print(f"\nLoaded {len(results_by_experiment)} datasets")

# Simulate a combined OmniMatch dataset
results_by_experiment["omnimatch_both_test"] = defaultdict(dict)

for exp_group, model_dfs in results_by_experiment["omnimatch_city_test"].items():
    for model_name, df_city in model_dfs.items():
        results_by_experiment["omnimatch_both_test"][exp_group][model_name] = pl.concat(
            [df_city, results_by_experiment["omnimatch_culture_test"][exp_group][model_name]]
        )
print("\nAdded omnimatch_both_test dataset")

omnimatch_city_test: centralized(13), all_views(4), 12_views(8)
webtable-union: centralized(8), 08_views(7)
omnimatch_culture_test: centralized(13), all_views(4), 12_views(8)
santos-union: centralized(13), all_views(4), 12_views(8)
freyja: centralized(13), all_views(4), 12_views(8)

Loaded 5 datasets

Added omnimatch_both_test dataset


In [6]:
def gather_centralized_stats(
    centralized_results: dict[str, pl.DataFrame], metric: str, k_vals: list[int]
) -> tuple[dict[int, list[float]], list[float]]:
    """Gather centralized and max recall stats for plotting."""
    model_means_per_k: dict[int, list[float]] = defaultdict(list)
    max_metric_vals: list[float] = []
    for model, df in centralized_results.items():
        if "starmie" in model or "deepjoin" in model:
            continue  # Skip the additional DeepJoin and Starmie finetuning results here
        for k in k_vals:
            col = f"{metric}@{k}"
            if col in df.columns:
                model_means_per_k[k].append(df[col].mean())  # pyright: ignore[reportArgumentType]
        if not max_metric_vals:
            max_metric_vals.extend(
                min(1.0, df[f"max_{metric}@{k}"].mean())  # pyright: ignore[reportArgumentType]
                for k in k_vals
            )
    return model_means_per_k, max_metric_vals


def add_centralized_and_max(
    ax: Axes, centralized_results: dict[str, pl.DataFrame], metric: str, k_vals: list[int]
) -> None:
    """Add centralized and max recall lines to the plot."""
    model_means_per_k, max_metric_vals = gather_centralized_stats(
        centralized_results, metric, k_vals
    )
    if any(model_means_per_k.values()):
        all_means = [np.mean(model_means_per_k[k]) for k in k_vals]
        all_mins = [np.min(model_means_per_k[k]) for k in k_vals]
        all_maxs = [np.max(model_means_per_k[k]) for k in k_vals]
        ax.fill_between(k_vals, all_mins, all_maxs, alpha=0.3, color=COLOR_MAP["centralized"])
        ax.plot(k_vals, all_means, color=COLOR_MAP["centralized"])
    if max_metric_vals and len(max_metric_vals) == len(k_vals):
        ax.plot(k_vals, max_metric_vals, "--", color=COLOR_MAP["max_metric"], alpha=0.8)

In [7]:
k_vals = [1, 5, 10, 15, 20, 25, 30, 35, 40]
for exp_id, datasets in [
    ("12_views", ["freyja", "omnimatch_both_test", "santos-union"]),
    ("08_views", ["webtable-union"]),
]:
    for dataset in datasets:
        exp_groups = results_by_experiment[dataset]
        if "centralized" not in exp_groups:
            print(f"{dataset}: skipping, no centralized results")
            continue

        for metric in ["precision", "recall"]:
            fig, ax = plt.subplots(1, 1, figsize=(4.8, 3.1), layout="tight")
            add_centralized_and_max(ax, exp_groups["centralized"], metric, k_vals)
            for model, df in exp_groups[exp_id].items():
                if model in {"cl_default", "cl_noval"}:
                    continue

                col = f"{metric}@{k_vals[0]}"
                if col not in df.columns:
                    print(f"Metric columns not found: {df.columns}")
                    continue
                means: list[float] = [df[f"{metric}@{k}"].mean() for k in k_vals]  # pyright: ignore[reportAssignmentType]
                ax.plot(k_vals, means, color=COLOR_MAP[model])

            ax.set_xlabel("k")
            ax.set_ylabel(f"{metric.capitalize()}@k")
            ax.set_xticks(k_vals)
            ax.set_xlim(min(k_vals) - 0.1, max(k_vals) + 0.1)
            ax.set_ylim(0, 1)

            fig.savefig(plots_dir / f"effectiveness-{dataset}-{metric}.svg")
            plt.close(fig)

## View Robustness Figure

In [8]:
# Load results per dataset and experiment group
robustness_path = Path("logs/view-robustness")

dataset_dfs: dict[str, pl.DataFrame] = {}
for dataset_path in robustness_path.iterdir():
    if not dataset_path.is_dir():
        continue
    dataset = dataset_path.name

    df_list: list[pl.DataFrame] = []
    for exp_group_path in dataset_path.iterdir():
        if not exp_group_path.is_dir():
            continue

        df_list.extend(
            pl.read_csv(f).with_columns(
                pl.lit(exp_group_path.name).alias("exp_group"),
                pl.lit(f.stem.split("-")[1]).alias("method"),
            )
            for f in exp_group_path.glob("*.csv")
        )

    dataset_dfs[dataset] = pl.concat(df_list).with_columns(pl.lit(dataset).alias("dataset"))

# Compute per-view average metrics (mean across queries per view)
per_view_means_dict: dict[str, pl.DataFrame] = {}
for dataset, dataset_df in dataset_dfs.items():
    metric_cols = [c for c in dataset_df.columns if "@" in c]
    per_view_mean = dataset_df.group_by(["exp_group", "dataset", "method", "view"]).agg(
        [pl.col(c).mean().alias(c) for c in metric_cols]
    )
    per_view_means_dict[dataset] = per_view_mean

In [ ]:
for k in [10, 20]:
    for exp_group, datasets in [
        ("12_views", ["freyja", "omnimatch_city_test", "omnimatch_culture_test", "santos-union"]),
        ("08_views", ["webtable-union"]),
    ]:
        for dataset in datasets:
            per_view_means = per_view_means_dict[dataset]
            if len(per_view_means) == 0:
                continue

            exp_df = per_view_means.filter(pl.col("exp_group") == exp_group)

            for metric in ["precision", "recall"]:
                fig, ax = plt.subplots(1, 1, figsize=(4.8, 3.45), layout="tight")

                max_metric_value: float = exp_df[f"max_{metric}@{k}"].max()  # type: ignore[arg-type]
                ax.hlines(
                    y=max_metric_value,
                    xmin=-0.5,
                    xmax=3.5,
                    linestyles="--",
                    colors=COLOR_MAP["max_metric"],
                    alpha=0.8,
                )

                sns.boxplot(
                    data=exp_df,
                    x="method",
                    y=f"{metric}@{k}",
                    ax=ax,
                    order=["cl_optim", "la2m_default", "la2m_nopca", "v2v"],
                    hue="method",
                    hue_order=["cl_optim", "la2m_default", "la2m_nopca", "v2v"],
                    palette=COLOR_MAP,
                    linewidth=1.5,
                )
                ax.set_xlabel("")
                ax.set_ylabel(f"{metric.capitalize()}@{k}")
                ax.set_ylim(0, 1)
                ax.set_xticks(
                    [0, 1, 2, 3], labels=["Fed\nAugment", "LA2M", "LA2M\n(no PCA)", "vec2vec"]
                )

                fig.savefig(plots_dir / f"view_robustness-{dataset}-{metric}-{k}.svg")
                plt.close(fig)

## View Count Figure

In [10]:
k_vals = [5, 10, 20]

all_views: dict[str, pl.DataFrame] = {}
centralized: dict[str, dict[str, dict[int, list[float]]]] = defaultdict(dict)
for dataset in ["freyja", "omnimatch_city_test", "omnimatch_culture_test", "santos-union"]:
    df_list = []
    for ckpt_name in (effectiveness_path / dataset / "all_views").iterdir():
        ckpt_params = ckpt_name.stem.split("-")
        model_name = ckpt_params[1]
        n_views = ckpt_params[2].split("=")[1]
        df = pl.read_csv(ckpt_name).with_columns(
            pl.lit(model_name).alias("model"), pl.lit(int(n_views)).alias("n_views")
        )
        df_list.append(df)

    all_views[dataset] = (
        pl.concat(df_list)
        .group_by(["model", "n_views"])
        .agg(
            [
                pl.col(f"{metric}@{k}").mean().alias(f"{metric}@{k}")
                for metric in ["precision", "recall", "max_precision", "max_recall"]
                for k in k_vals
            ]
        )
    )

    centralized[dataset]["precision"], _ = gather_centralized_stats(
        results_by_experiment[dataset]["centralized"], "precision", k_vals
    )
    centralized[dataset]["recall"], _ = gather_centralized_stats(
        results_by_experiment[dataset]["centralized"], "recall", k_vals
    )

print(f"Loaded {len(all_views)} datasets")

# Simulate a combined OmniMatch dataset
all_views["omnimatch_both_test"] = pl.concat(
    [all_views["omnimatch_city_test"], all_views["omnimatch_culture_test"]]
)
centralized["omnimatch_both_test"]["precision"], _ = gather_centralized_stats(
    results_by_experiment["omnimatch_both_test"]["centralized"], "precision", k_vals
)
centralized["omnimatch_both_test"]["recall"], _ = gather_centralized_stats(
    results_by_experiment["omnimatch_both_test"]["centralized"], "recall", k_vals
)
print("Added omnimatch_both_test dataset")

Loaded 4 datasets
Added omnimatch_both_test dataset


In [11]:
k = 10
for dataset in ["omnimatch_both_test", "santos-union"]:
    df = all_views[dataset]

    min_views: int = df["n_views"].min()  # type: ignore[arg-type]
    max_views: int = df["n_views"].max()  # type: ignore[arg-type]

    for metric in ["precision", "recall"]:
        fig, ax = plt.subplots(1, 1, figsize=(4.8, 3.1), layout="tight")
        max_metric: float = df[f"max_{metric}@{k}"].max()  # type: ignore[arg-type]
        centralized_metrics = centralized[dataset][metric][k]

        sns.lineplot(
            data=df,
            x="n_views",
            y=f"{metric}@{k}",
            hue="model",
            palette=COLOR_MAP,
            errorbar=None,
            legend=False,
            ax=ax,
        )
        ax.hlines(
            y=max_metric,
            xmin=min_views,
            xmax=max_views,
            linestyles="--",
            colors=COLOR_MAP["max_metric"],
            alpha=0.8,
        )
        ax.hlines(
            y=np.mean(centralized_metrics),
            xmin=min_views,
            xmax=max_views,
            colors=COLOR_MAP["centralized"],
        )
        ax.fill_between(
            x=[min_views, max_views],
            y1=min(centralized_metrics),
            y2=max(centralized_metrics),
            color=COLOR_MAP["centralized"],
            alpha=0.3,
        )

        ax.set_xlabel("Number of views")
        ax.set_ylabel(f"{metric.capitalize()}@{k}")
        ax.set_xlim(min_views, max_views)
        ax.set_ylim(0, min(1, math.ceil(max_metric * 10) / 10))
        ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=6))

        fig.savefig(plots_dir / f"view_count-{dataset}-{metric}-{k}.svg")
        plt.close(fig)

## Efficiency Figure

In [12]:
results_by_exp_group: dict[str, pl.DataFrame] = {}
for exp_group_path in Path("logs/efficiency").iterdir():
    if not exp_group_path.is_dir():
        continue

    df_list = [
        pl.read_csv(ckpt_path).with_columns(pl.lit(ckpt_path.name.split("-")[1]).alias("method"))
        for ckpt_path in exp_group_path.iterdir()
    ]

    results_by_exp_group[exp_group_path.name] = pl.concat(df_list)

In [13]:
exp_group = "12_views"

aggregated = (
    results_by_exp_group[exp_group]
    .group_by(["method", "batch_size"])
    .agg(
        [
            pl.mean("inference_time_ms").alias("mean_inference_time_ms"),
            pl.std("inference_time_ms").alias("std_inference_time_ms"),
            pl.min("inference_time_ms").alias("min_inference_time_ms"),
            pl.max("inference_time_ms").alias("max_inference_time_ms"),
            pl.mean("peak_gpu_memory_gb"),
        ]
    )
    .with_columns(
        ((pl.col("batch_size") * 1000) / (pl.col("mean_inference_time_ms"))).alias(
            "mean_throughput_qps"
        )
    )
    .sort(["method", "batch_size"])
)
batch_sizes: list[int] = (
    aggregated.select(pl.col("batch_size")).unique().to_series().sort().to_list()
)

In [14]:
fig, ax = plt.subplots(1, 1, figsize=(8, 3.25), layout="tight")

df = aggregated.filter(pl.col("method").is_in(["cl_optim", "la2m_default", "la2m_nopca", "v2v"]))
sns.lineplot(
    data=df,
    x="batch_size",
    y="mean_throughput_qps",
    hue="method",
    palette=COLOR_MAP,
    legend=False,
    ax=ax,
)

ax.set_xlabel("Batch size")
ax.set_ylabel("Throughput (queries/s)")
ax.set_xlim(min(batch_sizes), max(batch_sizes))
ax.set_ylim(1e3, 3.5 * 1e6)
ax.set_yscale("log")
ax.set_xscale("log")

fig.savefig(plots_dir / "efficiency.svg")
plt.close(fig)

## Legends

In [15]:
fig = plt.figure(figsize=(30, 3))
fig.legend(
    handles=[
        Line2D([0], [0], color=COLOR_MAP["cl_optim"], label="FedAugment"),
        Line2D([0], [0], color=COLOR_MAP["la2m_default"], label="LA2M"),
        Line2D([0], [0], color=COLOR_MAP["la2m_nopca"], label="LA2M (no PCA)"),
        Line2D([0], [0], color=COLOR_MAP["v2v"], label="vec2vec"),
        Line2D([0], [0], color=COLOR_MAP["union_minus"], label="Truncate"),
        Line2D([0], [0], color=COLOR_MAP["union_plus"], label="Pad"),
        Line2D([0], [0], color=COLOR_MAP["centralized"], label="Centralized (avg/min/max)"),
        Line2D(
            [0],
            [0],
            color=COLOR_MAP["max_metric"],
            linestyle="--",
            alpha=0.8,
            label="Max precision/recall",
        ),
    ],
    labels=None,
    loc="center",
    ncol=4,
)
fig.savefig(plots_dir / "legend_effectiveness.svg")
plt.close(fig)

In [16]:
fig = plt.figure(figsize=(30, 3))
fig.legend(
    handles=[
        Line2D([0], [0], color=COLOR_MAP["cl_optim"], label="FedAugment"),
        Line2D([0], [0], color=COLOR_MAP["la2m_default"], label="LA2M"),
        Line2D([0], [0], color=COLOR_MAP["la2m_nopca"], label="LA2M (no PCA)"),
        Line2D([0], [0], color=COLOR_MAP["v2v"], label="vec2vec"),
        Line2D([0], [0], color=COLOR_MAP["centralized"], label="Centralized (avg/min/max)"),
        Line2D(
            [0],
            [0],
            color=COLOR_MAP["max_metric"],
            linestyle="--",
            alpha=0.8,
            label="Max precision/recall",
        ),
    ],
    labels=None,
    loc="center",
    ncol=3,
)
fig.savefig(plots_dir / "legend_view_count.svg")
plt.close(fig)

In [17]:
fig = plt.figure(figsize=(30, 3))
fig.legend(
    handles=[
        Line2D([0], [0], color=COLOR_MAP["cl_optim"], label="FedAugment"),
        Line2D([0], [0], color=COLOR_MAP["la2m_default"], label="LA2M"),
        Line2D([0], [0], color=COLOR_MAP["la2m_nopca"], label="LA2M (no PCA)"),
        Line2D([0], [0], color=COLOR_MAP["v2v"], label="vec2vec"),
    ],
    labels=None,
    loc="center",
    ncol=4,
)
fig.savefig(plots_dir / "legend_efficiency.svg")
plt.close(fig)